In [0]:
%run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_3", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_3", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
# git-lfs is required: the GR00T repo ships prebuilt wheels
# (scripts/deployment/dgpu/wheels/*.whl) as Git LFS objects. Without lfs they
# clone as text pointer files, and uv fails to read their metadata with
# "Invalid zip file structure" during sync (required-environments forces uv to
# inspect both the x86_64 and aarch64 wheels regardless of the live arch).
sudo apt-get install -y git-lfs;
git lfs install --skip-repo;
cd $HOME;
if [ ! -d Isaac-GR00T ]; then
  git clone --recurse-submodules https://github.com/NVIDIA/Isaac-GR00T;
else
  cd $HOME/Isaac-GR00T && git fetch;
fi;
# Pull the actual LFS blobs (no-op if already present).
cd $HOME/Isaac-GR00T && git lfs pull;
echo "== wheels (should be real archives, MBs not bytes) ==";
ls -la $HOME/Isaac-GR00T/scripts/deployment/dgpu/wheels/ 2>/dev/null || ls -la $HOME/Isaac-GR00T;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
cd $HOME/Isaac-GR00T;
uv sync --python 3.10;
uv pip install -e .;
sudo apt install -y ffmpeg libavcodec-dev libavformat-dev libavutil-dev libswscale-dev;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
cd $HOME/Isaac-GR00T;
uv pip install --upgrade pip;
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124;
echo == Torch CUDA check ==;
uv run python -c 'import torch; print("torch:", torch.__version__); print("cuda available:", torch.cuda.is_available())';
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP  << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export CUDA_HOME=/usr/local/cuda-12.4;
export TORCH_CUDA_ARCH_LIST="8.0";
export MAX_JOBS=$(nproc);
cd $HOME/Isaac-GR00T;
uv pip install ninja packaging;
uv pip install flash-attn --no-build-isolation --no-cache-dir;
echo == flash-attn import check ==;
uv run python -c 'import flash_attn; print("flash_attn OK:", flash_attn.__version__)';
echo == Setup complete ==;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
cd $HOME/Isaac-GR00T;
uv run hf download nvidia/GR00T-N1.7-3B;
EOF

In [0]:
%skip
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
cd $HOME/Isaac-GR00T;
mkdir -p finetuned_models;
ls -la finetuned_models;
EOF